# Índices de Riesgo de Corrupción 2021 — Subasta Inversa Electrónica (SIE)

Este notebook reproduce el pipeline completo para las preguntas del año 2021:

1. Carga y limpieza de `preguntas_2021.csv`
2. Generación de embeddings con `text-embedding-3-large` (OpenAI), reutilizando el caché existente
3. Entrenamiento de modelos ML (NB, RF, LR) con los datos de entrenamiento existentes
4. Cálculo de probabilidades por pregunta
5. Cálculo de índices de riesgo por proceso (IC_M1_nb/rf/lr, IC_M2_nb/rf/lr, IC_M3, IC_M4)
6. Distribuciones diarias y semanales: Kapak (`nro_preguntas_acusatorias`) vs cada índice

Las salidas se guardan en `output/2021/`.

In [1]:
import sys
!{sys.executable} -m pip install python-dotenv openai 

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

In [2]:
# ─── Rutas ────────────────────────────────────────────────────────────────────
_cwd = Path.cwd().resolve()
BASE_DIR  = _cwd.parent if _cwd.name.lower() == 'codes' else _cwd
CODES_DIR = BASE_DIR / 'Codes'

DATA_2021_PATH = CODES_DIR / 'preguntas_2021.csv'
OUTPUT_DIR     = CODES_DIR / 'output' / '2021'
CACHE_PATH     = CODES_DIR / 'output' / 'embedding_cache_text-embedding-3-large.jsonl'

TRAIN_NB_PATH  = CODES_DIR / 'Archivos_Naive_Bayes'         / 'train_embeddings_balanced_AUG_GPT4o_mini_total_augmented_text-embedding-3-large.csv'
TRAIN_RF_PATH  = CODES_DIR / 'Archivos_Random_Forest'       / 'train_embeddings_balanced_total_sentence_prompt_GPT4o_mini_text-embedding-3-large.csv'
TRAIN_LR_PATH  = CODES_DIR / 'Archivos_Logistic_Regression' / 'train_embeddings_balanced_total_sinonimos_text-embedding-3-large.csv'

EMBED_MODEL = 'text-embedding-3-large'
BATCH_SIZE  = 100
YEAR_FILTER = 2021

# Pesos ponderados por Brier score (de modelos_probabilidades.ipynb)
W_LR = 0.5478240336268073
W_RF = 0.3998231918002441
W_NB = 0.05235277457294863

INDEX_COLUMNS = ['IC_M1_nb','IC_M1_rf','IC_M1_lr',
                  'IC_M2_nb','IC_M2_rf','IC_M2_lr',
                  'IC_M3','IC_M4']
KAPAK_COL = 'nro_preguntas_acusatorias'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── API Key ───────────────────────────────────────────────────────────────────
load_dotenv(BASE_DIR / '.env')
load_dotenv(CODES_DIR / '.env')

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '').strip()
if not OPENAI_API_KEY:
    raise ValueError(
        'OPENAI_API_KEY no encontrada. '
        'Agrega OPENAI_API_KEY=sk-... a tu archivo .env en la raiz del proyecto.'
    )

print(f'BASE_DIR   : {BASE_DIR}')
print(f'CODES_DIR  : {CODES_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'Modelo     : {EMBED_MODEL}')

BASE_DIR   : D:\Noveno Semestre\Tesis Informal
CODES_DIR  : D:\Noveno Semestre\Tesis Informal\Codes
OUTPUT_DIR : D:\Noveno Semestre\Tesis Informal\Codes\output\2021
Modelo     : text-embedding-3-large


## 1. Carga y preparación de datos 2021

In [3]:
raw_df = pd.read_csv(DATA_2021_PATH)
print(f'Filas totales cargadas : {len(raw_df):,}')
print(f'Columnas               : {raw_df.columns.tolist()}')
raw_df.head(3)

Filas totales cargadas : 134,294
Columnas               : ['contract_id,"codigo","fecha_publicacion","estado_del_proceso","presupuesto_referencial_total_sin_iva","monto_contrato","monto_adjudicacion","valor_preguntas","es_acusatoria","clasificacion"']


,"contract_id,""codigo"",""fecha_publicacion"",""estado_del_proceso"",""presupuesto_referencial_total_sin_iva"",""monto_contrato"",""monto_adjudicacion"",""valor_preguntas"",""es_acusatoria"",""clasificacion"""
0,"1608622,""SIE-CZ9S-006-2020"",""2021-01-04"",""Fina..."
1,"1608622,""SIE-CZ9S-006-2020"",""2021-01-04"",""Fina..."
2,"1608622,""SIE-CZ9S-006-2020"",""2021-01-04"",""Fina..."


In [4]:
import csv as _csv
import io
_rows = []
with open(DATA_2021_PATH, 'r', encoding='utf-8', newline='') as _f:
    for _row in _csv.reader(_f):
        _rows.append(_row[0])   # peel the outer quote layer
raw_df = pd.read_csv(io.StringIO('\n'.join(_rows)))

In [5]:
df = raw_df.copy()

# Texto de la pregunta
df['valor_preguntas'] = df['valor_preguntas'].fillna('').astype(str).str.strip()

# Normalizar etiqueta acusatoria a booleano
df['es_acusatoria'] = (
    df['es_acusatoria']
    .astype(str).str.strip().str.lower()
    .map({'true': True, 'false': False, '1': True, '0': False})
    .fillna(False)
)

# Fecha de publicacion como datetime
df['fecha_publicacion'] = pd.to_datetime(df['fecha_publicacion'], errors='coerce')

# Filtrar filas sin texto o sin fecha valida
n_before = len(df)
df = df[df['valor_preguntas'].str.len() > 0].copy()
df = df[df['fecha_publicacion'].notna()].copy()
df = df.reset_index(drop=True)

print(f'Filas eliminadas (sin texto/fecha) : {n_before - len(df):,}')
print(f'Filas validas                      : {len(df):,}')
print(f'Procesos unicos                    : {df["contract_id"].nunique():,}')
print(f'Preguntas acusatorias              : {int(df["es_acusatoria"].sum()):,} ({df["es_acusatoria"].mean()*100:.1f}%)')
print(f'Rango de fechas                    : {df["fecha_publicacion"].min().date()} -> {df["fecha_publicacion"].max().date()}')
df.head(3)

Filas eliminadas (sin texto/fecha) : 0
Filas validas                      : 134,294
Procesos unicos                    : 17,152
Preguntas acusatorias              : 5,709 (4.3%)
Rango de fechas                    : 2021-01-04 -> 2021-12-30


,contract_id,codigo,fecha_publicacion,estado_del_proceso,presupuesto_referencial_total_sin_iva,monto_contrato,monto_adjudicacion,valor_preguntas,es_acusatoria,clasificacion
0,1608622,SIE-CZ9S-006-2020,2021-01-04,Finalizada,"$232,534.00","$176,995.00","$176,995.00","""Señores, en la Experiencia tecnica del oferen...",False,No Acusatoria
1,1608622,SIE-CZ9S-006-2020,2021-01-04,Finalizada,"$232,534.00","$176,995.00","$176,995.00","""Gracias por la invitación, para dar cumplimie...",False,No Acusatoria
2,1608622,SIE-CZ9S-006-2020,2021-01-04,Finalizada,"$232,534.00","$176,995.00","$176,995.00","""Buen día, se puede participar ÚNICAMENTE con ...",False,No Acusatoria


## 2. Generación de embeddings con `text-embedding-3-large`

Se reutiliza el caché existente (`embedding_cache_text-embedding-3-large.jsonl`) para evitar llamadas redundantes a la API.
Solo se llamara a la API para textos nuevos que no esten en el caché.

> **Nota:** Con un dataset grande, el primer paso puede tomar varios minutos dependiendo de cuantos textos sean nuevos.

In [6]:
def load_cache(cache_path: Path) -> dict:
    """Carga el cache de embeddings desde archivo JSONL."""
    cache = {}
    if not cache_path.exists():
        return cache
    with cache_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            cache[item['text']] = item['embedding']
    return cache


def save_cache(cache_path: Path, cache: dict) -> None:
    """Guarda el cache de embeddings a archivo JSONL (ordenado por texto)."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open('w', encoding='utf-8') as f:
        for key in sorted(cache.keys()):
            f.write(json.dumps({'text': key, 'embedding': cache[key]}, ensure_ascii=False) + '\n')


def embed_texts(
    texts: list,
    api_key: str,
    model: str = EMBED_MODEL,
    batch_size: int = BATCH_SIZE,
    cache_path: Path = CACHE_PATH,
) -> list:
    """Genera embeddings usando cache. Solo llama la API para textos nuevos."""
    client = OpenAI(api_key=api_key)
    cache = load_cache(cache_path)

    unique_texts = sorted(set(texts))
    missing = [t for t in unique_texts if t not in cache]

    print(f'Textos unicos    : {len(unique_texts):,}')
    print(f'En cache         : {len(unique_texts) - len(missing):,}')
    print(f'Nuevos a embedir : {len(missing):,}')

    if missing:
        n_batches = (len(missing) + batch_size - 1) // batch_size
        for i in range(0, len(missing), batch_size):
            batch_idx = i // batch_size + 1
            chunk = missing[i: i + batch_size]
            retries = 0
            while True:
                try:
                    resp = client.embeddings.create(model=model, input=chunk)
                    for text_in, item in zip(chunk, resp.data):
                        cache[text_in] = item.embedding
                    break
                except Exception as exc:
                    retries += 1
                    if retries > 5:
                        raise RuntimeError(f'Error en lote {batch_idx}: {exc}') from exc
                    wait = min(2 ** retries, 30)
                    print(f'  [WARN] Lote {batch_idx} fallo. Reintentando en {wait}s...')
                    time.sleep(wait)
            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f'  Lote {batch_idx}/{n_batches} completado.')
        save_cache(cache_path, cache)
        print(f'Cache actualizado: {cache_path}')

    return [cache[t] for t in texts]

In [7]:
# ── Configuracion de chunk ──────────────────────────────────────────────────
# Cambia START_IDX para procesar el siguiente bloque de 5 000 preguntas.
#   Ejemplo:      0  →  preguntas      0 –  4 999
#             5 000  →  preguntas  5 000 –  9 999
#            10 000  →  preguntas 10 000 – 14 999  ... y así sucesivamente
START_IDX  = 75000        # <-- Modifica este valor manualmente antes de ejecutar
CHUNK_SIZE = 5000

end_idx     = min(START_IDX + CHUNK_SIZE, len(df))
texts_chunk = df['valor_preguntas'].iloc[START_IDX:end_idx].tolist()

print(f'Total de preguntas en df : {len(df):,}')
print(f'Procesando chunk         : [{START_IDX:,} – {end_idx - 1:,}]  ({len(texts_chunk):,} preguntas)')
embeddings_chunk = embed_texts(texts_chunk, OPENAI_API_KEY)
print(f'Embeddings listos : {len(embeddings_chunk):,}')


Total de preguntas en df : 134,294
Procesando chunk         : [75,000 – 79,999]  (5,000 preguntas)
Textos unicos    : 4,186
En cache         : 245
Nuevos a embedir : 3,941
  Lote 20/40 completado.
  Lote 40/40 completado.
Cache actualizado: D:\Noveno Semestre\Tesis Informal\Codes\output\embedding_cache_text-embedding-3-large.jsonl
Embeddings listos : 5,000


In [8]:
emb_chunk = pd.DataFrame(embeddings_chunk)
emb_chunk.columns = [str(i) for i in range(emb_chunk.shape[1])]

chunk_filename = f'sie_2021_embeddings_chunk_{START_IDX:06d}_{end_idx:06d}.csv'
chunk_path     = OUTPUT_DIR / chunk_filename
emb_chunk.to_csv(chunk_path, index=False)

print(f'Chunk guardado : {chunk_path}')
print(f'  Filas (preguntas) : {emb_chunk.shape[0]:,}')
print(f'  Columnas (dims)   : {emb_chunk.shape[1]}')

# Mostrar estado de chunks existentes
existing_chunks = sorted(OUTPUT_DIR.glob('sie_2021_embeddings_chunk_*.csv'))
covered = sum(
    int(p.stem.split('_')[-1]) - int(p.stem.split('_')[-2])
    for p in existing_chunks
)
print(f'Chunks existentes ({len(existing_chunks)}): {covered:,} / {len(df):,} preguntas cubiertas')
for p in existing_chunks:
    parts = p.stem.split('_')
    print(f'  {p.name}  ({int(parts[-1]) - int(parts[-2]):,} filas)')


Chunk guardado : D:\Noveno Semestre\Tesis Informal\Codes\output\2021\sie_2021_embeddings_chunk_075000_080000.csv
  Filas (preguntas) : 5,000
  Columnas (dims)   : 3072
Chunks existentes (16): 80,000 / 134,294 preguntas cubiertas
  sie_2021_embeddings_chunk_000000_005000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_005000_010000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_010000_015000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_015000_020000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_020000_025000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_025000_030000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_030000_035000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_035000_040000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_040000_045000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_045000_050000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_050000_055000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_055000_060000.csv  (5,000 filas)
  sie_2021_embeddings_chunk_060000_0650

In [9]:
import pandas as pd

my_df = pd.read_csv("output/2021/sie_2021_embeddings_chunk_070000_075000.csv")
print(my_df.shape[0])

5000


In [10]:
# ── Ensamblar emb_matrix desde todos los chunks disponibles ─────────────────
# Ejecuta esta celda UNA VEZ que hayas generado TODOS los chunks.
# Valida que los chunks cubran consecutivamente todas las preguntas de df.

chunk_files = sorted(OUTPUT_DIR.glob('sie_2021_embeddings_chunk_*.csv'))
if not chunk_files:
    raise FileNotFoundError(
        f'No se encontraron chunks en {OUTPUT_DIR}. '
        'Genera los chunks primero ejecutando las celdas anteriores.'
    )

print(f'Cargando {len(chunk_files)} chunk(s)...')
parts_list = []
expected_start = 0
for p in chunk_files:
    stem_parts  = p.stem.split('_')
    chunk_start = int(stem_parts[-2])
    chunk_end   = int(stem_parts[-1])
    if chunk_start != expected_start:
        raise ValueError(
            f'Falta chunk: se esperaba inicio en {expected_start:,}, '
            f'pero el siguiente archivo comienza en {chunk_start:,}, '
            f'Genera el chunk [{expected_start:,} – {chunk_start - 1:,}] antes de continuar.'
        )
    chunk_df = pd.read_csv(p)
    parts_list.append(chunk_df)
    print(f'  {p.name}  → {len(chunk_df):,} filas')
    expected_start = chunk_end

emb_matrix = pd.concat(parts_list, ignore_index=True)
emb_matrix.columns = [str(i) for i in range(emb_matrix.shape[1])]

if len(emb_matrix) != len(df):
    raise ValueError(
        f'La matriz ensamblada tiene {len(emb_matrix):,} filas '
        f'pero df tiene {len(df):,} preguntas. '
        'Asegúrate de que todos los chunks estén generados.'
    )

print(f'emb_matrix ensamblada correctamente.')
print(f'  Dimensiones : {emb_matrix.shape}')
print(f'  Filas       : {emb_matrix.shape[0]:,}')
print(f'  Columnas    : {emb_matrix.shape[1]}')


Cargando 16 chunk(s)...
  sie_2021_embeddings_chunk_000000_005000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_005000_010000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_010000_015000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_015000_020000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_020000_025000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_025000_030000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_030000_035000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_035000_040000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_040000_045000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_045000_050000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_050000_055000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_055000_060000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_060000_065000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_065000_070000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_070000_075000.csv  → 5,000 filas
  sie_2021_embeddings_chunk_075000_080000.csv  → 5,000 filas


ValueError: La matriz ensamblada tiene 80,000 filas pero df tiene 134,294 preguntas. Asegúrate de que todos los chunks estén generados.

## 3. Entrenamiento de modelos y probabilidades por pregunta

Se entrenan los tres modelos seleccionados en la tesis (NB, RF, LR) con sus respectivos datasets de entrenamiento.
Los hiperparametros son los mismos usados en `kapak_real_process_indices.ipynb`.

In [ ]:
def prepare_model_inputs(train_path: Path, target_emb: pd.DataFrame):
    """Carga training CSV y alinea columnas de features con el target."""
    train = pd.read_csv(train_path)
    features = [c for c in train.columns if c != 'label']
    missing_cols = [c for c in features if c not in target_emb.columns]
    if missing_cols:
        raise ValueError(
            f'Faltan {len(missing_cols)} columnas de embedding en el target: {missing_cols[:5]}...'
        )
    x_train  = train[features].to_numpy()
    y_train  = train['label'].to_numpy()
    x_target = target_emb[features].to_numpy()
    return x_train, y_train, x_target


# ── Naive Bayes ────────────────────────────────────────────────────────────────
print('Entrenando Naive Bayes...')
x_train_nb, y_train_nb, x_nb = prepare_model_inputs(TRAIN_NB_PATH, emb_matrix)
model_nb = GaussianNB(var_smoothing=0.35111917342151305)
model_nb.fit(x_train_nb, y_train_nb)
prob_nb = model_nb.predict_proba(x_nb)[:, 1]
print(f'  NB listo  | prob in [{prob_nb.min():.4f}, {prob_nb.max():.4f}] | train size: {len(x_train_nb):,}')

# ── Random Forest ──────────────────────────────────────────────────────────────
print('Entrenando Random Forest (puede tardar unos minutos)...')
x_train_rf, y_train_rf, x_rf = prepare_model_inputs(TRAIN_RF_PATH, emb_matrix)
model_rf = RandomForestClassifier(
    max_depth=None, min_samples_leaf=1, n_estimators=200, random_state=42, n_jobs=-1
)
model_rf.fit(x_train_rf, y_train_rf)
prob_rf = model_rf.predict_proba(x_rf)[:, 1]
print(f'  RF listo  | prob in [{prob_rf.min():.4f}, {prob_rf.max():.4f}] | train size: {len(x_train_rf):,}')

# ── Logistic Regression ────────────────────────────────────────────────────────
print('Entrenando Logistic Regression...')
x_train_lr, y_train_lr, x_lr = prepare_model_inputs(TRAIN_LR_PATH, emb_matrix)
model_lr = LogisticRegression(C=100, penalty='l2', solver='saga', max_iter=1000, random_state=42)
model_lr.fit(x_train_lr, y_train_lr)
prob_lr = model_lr.predict_proba(x_lr)[:, 1]
print(f'  LR listo  | prob in [{prob_lr.min():.4f}, {prob_lr.max():.4f}] | train size: {len(x_train_lr):,}')

print('\nModelos entrenados correctamente.')

In [ ]:
q_probs = df[['contract_id', 'fecha_publicacion', 'es_acusatoria']].copy().reset_index(drop=True)
q_probs['prob_nb'] = prob_nb
q_probs['prob_rf'] = prob_rf
q_probs['prob_lr'] = prob_lr

q_probs_path = OUTPUT_DIR / 'sie_2021_question_probabilities.csv'
q_probs.to_csv(q_probs_path, index=False)
print(f'Probabilidades por pregunta guardadas: {q_probs_path}')
q_probs.head(10)

## 4. Índices de riesgo por proceso (IC_M1 – IC_M4)

**Track A** — cada modelo opera de forma independiente:
- `IC_M1_nb/rf/lr`: media aritmética de probabilidades del modelo por proceso
- `IC_M2_nb/rf/lr`: Noisy-OR de probabilidades del modelo por proceso: $1 - \prod_i(1-p_i)$

**Track B** — soft vote ponderado por Brier score:
- `IC_M3`: media del soft vote ponderado $(w_{LR}\cdot p_{LR} + w_{RF}\cdot p_{RF} + w_{NB}\cdot p_{NB})$
- `IC_M4`: Noisy-OR del soft vote ponderado

In [ ]:
def noisy_or(arr: np.ndarray) -> float:
    """Noisy-OR: 1 - prod(1 - p_i). Acumulacion probabilistica de riesgo."""
    if arr.size == 0:
        return float('nan')
    return float(1.0 - np.prod(1.0 - np.clip(arr, 0.0, 1.0)))


def compute_process_indices(df_q: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula 8 indices de riesgo por proceso.
    Track A: IC_M1_nb/rf/lr (media), IC_M2_nb/rf/lr (Noisy-OR)
    Track B: IC_M3 (soft vote ponderado -> media), IC_M4 (soft vote ponderado -> Noisy-OR)
    """
    rows = []
    for contract_id, g in df_q.groupby('contract_id', sort=True):
        p_nb = g['prob_nb'].to_numpy(dtype=float)
        p_rf = g['prob_rf'].to_numpy(dtype=float)
        p_lr = g['prob_lr'].to_numpy(dtype=float)
        r_q  = W_LR * p_lr + W_RF * p_rf + W_NB * p_nb  # soft vote ponderado por Brier

        rows.append({
            'contract_id': contract_id,
            'n_preguntas': len(g),
            # Track A — media aritmetica por modelo
            'IC_M1_nb'  : float(p_nb.mean()),
            'IC_M1_rf'  : float(p_rf.mean()),
            'IC_M1_lr'  : float(p_lr.mean()),
            # Track A — Noisy-OR por modelo
            'IC_M2_nb'  : noisy_or(p_nb),
            'IC_M2_rf'  : noisy_or(p_rf),
            'IC_M2_lr'  : noisy_or(p_lr),
            # Track B — soft vote ponderado
            'IC_M3'     : float(r_q.mean()),
            'IC_M4'     : noisy_or(r_q),
        })
    return pd.DataFrame(rows)


indices_df = compute_process_indices(q_probs)

indices_path = OUTPUT_DIR / 'sie_2021_process_indices.csv'
indices_df.to_csv(indices_path, index=False)

print(f'Procesos con indices calculados: {len(indices_df):,}')
print(f'Indices guardados en: {indices_path}')
indices_df.head(10)

## 5. Etiquetas Kapak a nivel de proceso

Se agrega `es_acusatoria` desde las preguntas individuales a nivel de proceso:
- `nro_preguntas_acusatorias`: count de preguntas acusatorias en el proceso
- `label_acusatorio`: 1 si hay al menos una pregunta acusatoria, 0 si no

In [ ]:
labels_df = (
    df.groupby('contract_id', as_index=False)
    .agg(
        fecha_publicacion          = ('fecha_publicacion', 'first'),
        nro_preguntas_acusatorias  = ('es_acusatoria', 'sum'),
        n_preguntas_total          = ('es_acusatoria', 'count'),
    )
)
labels_df['label_acusatorio']          = (labels_df['nro_preguntas_acusatorias'] > 0).astype(int)
labels_df['nro_preguntas_acusatorias'] = labels_df['nro_preguntas_acusatorias'].astype(int)

labels_path = OUTPUT_DIR / 'sie_2021_process_labels.csv'
labels_df.to_csv(labels_path, index=False)

print(f'Procesos totales                     : {len(labels_df):,}')
print(f'Con >= 1 pregunta acusatoria (label=1): {labels_df["label_acusatorio"].sum():,}')
print(f'Total preguntas acusatorias           : {labels_df["nro_preguntas_acusatorias"].sum():,}')
print(f'Labels guardados en: {labels_path}')
labels_df.head(10)

In [ ]:
# Unir etiquetas con indices de riesgo
merged = labels_df.merge(
    indices_df[['contract_id'] + INDEX_COLUMNS],
    on='contract_id', how='inner'
)
merged = merged[merged['fecha_publicacion'].notna()].copy()

print(f'Procesos en el merge    : {len(merged):,}')
print(f'Rango de anos           : {merged["fecha_publicacion"].dt.year.min()} - {merged["fecha_publicacion"].dt.year.max()}')
print(f'Rango de fechas         : {merged["fecha_publicacion"].min().date()} -> {merged["fecha_publicacion"].max().date()}')
merged.head(5)

## 6. Distribuciones diarias: Kapak vs Índices de Riesgo

Para cada uno de los 8 indices se genera:
- Un grafico de barras diario individual (PNG)
- Un archivo CSV con los agregados diarios
- Un panel 2x4 con todos los indices juntos

In [ ]:
daily_output = OUTPUT_DIR / 'daily'
daily_output.mkdir(parents=True, exist_ok=True)

data_plot = merged.copy()
data_plot['day_bucket'] = data_plot['fecha_publicacion'].dt.normalize()

summary_daily = []
n_idx = len(INDEX_COLUMNS)
fig, axes = plt.subplots(2, 4, figsize=(28, 10), sharex=False)
axes_flat = axes.flatten()

for i, idx_col in enumerate(INDEX_COLUMNS):
    daily = (
        data_plot.groupby('day_bucket', as_index=False)
        .agg(
            kapak_sum_day   = (KAPAK_COL, 'sum'),
            index_sum_day   = (idx_col, 'sum'),
            n_processes_day = ('contract_id', 'nunique'),
        )
        .sort_values('day_bucket')
    )

    csv_path = daily_output / f'daily_aggregate_{idx_col}.csv'
    daily.to_csv(csv_path, index=False)

    x     = np.arange(len(daily))
    width = 0.42
    step  = max(1, len(x) // 20)

    # Figura individual
    sfig, sax = plt.subplots(figsize=(16, 6))
    sax.bar(x - width/2, daily['kapak_sum_day'], width=width,
            label=f'Kapak ({KAPAK_COL})', color='#4472C4')
    sax.bar(x + width/2, daily['index_sum_day'], width=width,
            label=f'Indice ({idx_col})',  color='#ED7D31')
    sax.set_title(f'Distribucion Diaria 2021: Kapak vs {idx_col}', fontsize=13)
    sax.set_xlabel('Dia')
    sax.set_ylabel('Suma diaria')
    sax.grid(True, alpha=0.3, axis='y')
    sax.legend()
    sax.set_xticks(x[::step])
    sax.set_xticklabels(
        daily['day_bucket'].iloc[::step].dt.strftime('%Y-%m-%d'),
        rotation=45, ha='right'
    )
    sfig.tight_layout()
    fig_path = daily_output / f'daily_comparison_{idx_col}.png'
    sfig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(sfig)

    corr = (daily['kapak_sum_day'].corr(daily['index_sum_day'])
            if len(daily) > 1 else float('nan'))
    summary_daily.append({
        'index_column'            : idx_col,
        'days'                    : len(daily),
        'kapak_total_sum'         : daily['kapak_sum_day'].sum(),
        'kapak_mean_daily_sum'    : daily['kapak_sum_day'].mean(),
        'index_total_sum'         : daily['index_sum_day'].sum(),
        'index_mean_daily_sum'    : daily['index_sum_day'].mean(),
        'pearson_corr_with_kapak' : corr,
    })

    # Panel
    ax = axes_flat[i]
    step_panel = max(1, len(x) // 8)
    ax.bar(x - width/2, daily['kapak_sum_day'], width=width, label='Kapak', color='#4472C4')
    ax.bar(x + width/2, daily['index_sum_day'], width=width, label=idx_col,  color='#ED7D31')
    ax.set_title(idx_col, fontsize=10)
    ax.grid(True, alpha=0.25, axis='y')
    ax.set_xticks(x[::step_panel])
    ax.set_xticklabels(
        daily['day_bucket'].iloc[::step_panel].dt.strftime('%m-%d'),
        rotation=45, ha='right', fontsize=7
    )

for j in range(n_idx, len(axes_flat)):
    axes_flat[j].axis('off')

handles, labels_leg = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='upper center', ncol=4, frameon=False, fontsize=10)
fig.suptitle('Distribuciones Diarias 2021: Kapak vs Indices de Riesgo', fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.94))
multi_path = daily_output / 'daily_comparison_all_indices_2x4.png'
fig.savefig(multi_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

summary_daily_df = pd.DataFrame(summary_daily)
summary_daily_df.to_csv(daily_output / 'daily_indices_summary.csv', index=False)
print(f'[OK] Panel diario: {multi_path}')
print(f'[OK] Figuras y CSVs individuales guardados en: {daily_output}')
summary_daily_df

## 7. Distribuciones semanales: Kapak vs Índices de Riesgo

Agregacion por semana (lunes de cada semana ISO como bucket). Misma estructura que la seccion anterior.

In [ ]:
weekly_output = OUTPUT_DIR / 'weekly'
weekly_output.mkdir(parents=True, exist_ok=True)

# Bucket semanal: lunes de la semana de cada fecha
dow = data_plot['fecha_publicacion'].dt.dayofweek  # 0 = lunes
data_plot['week_bucket'] = (data_plot['fecha_publicacion'] - pd.to_timedelta(dow, unit='D')).dt.normalize()

summary_weekly = []
fig_w, axes_w = plt.subplots(2, 4, figsize=(28, 10), sharex=False)
axes_w_flat = axes_w.flatten()

for i, idx_col in enumerate(INDEX_COLUMNS):
    weekly = (
        data_plot.groupby('week_bucket', as_index=False)
        .agg(
            kapak_sum_week   = (KAPAK_COL, 'sum'),
            index_sum_week   = (idx_col, 'sum'),
            n_processes_week = ('contract_id', 'nunique'),
        )
        .sort_values('week_bucket')
    )

    csv_path = weekly_output / f'weekly_aggregate_{idx_col}.csv'
    weekly.to_csv(csv_path, index=False)

    x     = np.arange(len(weekly))
    width = 0.42
    step  = max(1, len(x) // 20)

    # Figura individual
    sfig, sax = plt.subplots(figsize=(16, 6))
    sax.bar(x - width/2, weekly['kapak_sum_week'], width=width,
            label=f'Kapak ({KAPAK_COL})', color='#4472C4')
    sax.bar(x + width/2, weekly['index_sum_week'], width=width,
            label=f'Indice ({idx_col})',   color='#ED7D31')
    sax.set_title(f'Distribucion Semanal 2021: Kapak vs {idx_col}', fontsize=13)
    sax.set_xlabel('Semana (inicio de semana)')
    sax.set_ylabel('Suma semanal')
    sax.grid(True, alpha=0.3, axis='y')
    sax.legend()
    sax.set_xticks(x[::step])
    sax.set_xticklabels(
        weekly['week_bucket'].iloc[::step].dt.strftime('%Y-%m-%d'),
        rotation=45, ha='right'
    )
    sfig.tight_layout()
    fig_path = weekly_output / f'weekly_comparison_{idx_col}.png'
    sfig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(sfig)

    corr = (weekly['kapak_sum_week'].corr(weekly['index_sum_week'])
            if len(weekly) > 1 else float('nan'))
    summary_weekly.append({
        'index_column'             : idx_col,
        'weeks'                    : len(weekly),
        'kapak_total_sum'          : weekly['kapak_sum_week'].sum(),
        'kapak_mean_weekly_sum'    : weekly['kapak_sum_week'].mean(),
        'index_total_sum'          : weekly['index_sum_week'].sum(),
        'index_mean_weekly_sum'    : weekly['index_sum_week'].mean(),
        'pearson_corr_with_kapak'  : corr,
    })

    # Panel
    ax = axes_w_flat[i]
    step_panel = max(1, len(x) // 8)
    ax.bar(x - width/2, weekly['kapak_sum_week'], width=width, label='Kapak', color='#4472C4')
    ax.bar(x + width/2, weekly['index_sum_week'], width=width, label=idx_col,  color='#ED7D31')
    ax.set_title(idx_col, fontsize=10)
    ax.grid(True, alpha=0.25, axis='y')
    ax.set_xticks(x[::step_panel])
    ax.set_xticklabels(
        weekly['week_bucket'].iloc[::step_panel].dt.strftime('%m-%d'),
        rotation=45, ha='right', fontsize=7
    )

for j in range(n_idx, len(axes_w_flat)):
    axes_w_flat[j].axis('off')

handles_w, labels_w_leg = axes_w_flat[0].get_legend_handles_labels()
fig_w.legend(handles_w, labels_w_leg, loc='upper center', ncol=4, frameon=False, fontsize=10)
fig_w.suptitle('Distribuciones Semanales 2021: Kapak vs Indices de Riesgo', fontsize=15)
fig_w.tight_layout(rect=(0, 0, 1, 0.94))
multi_path_w = weekly_output / 'weekly_comparison_all_indices_2x4.png'
fig_w.savefig(multi_path_w, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig_w)

summary_weekly_df = pd.DataFrame(summary_weekly)
summary_weekly_df.to_csv(weekly_output / 'weekly_indices_summary.csv', index=False)
print(f'[OK] Panel semanal: {multi_path_w}')
print(f'[OK] Figuras y CSVs individuales guardados en: {weekly_output}')
summary_weekly_df

## 8. Resumen estadístico

In [ ]:
print('=== Estadisticas descriptivas de los indices de proceso (2021) ===')
display(indices_df[INDEX_COLUMNS].describe().round(4))

In [ ]:
print('=== Correlacion de Pearson con Kapak — Agregacion DIARIA ===')
display(
    summary_daily_df[['index_column','days','kapak_total_sum','index_total_sum','pearson_corr_with_kapak']]
    .round(4)
)

In [ ]:
print('=== Correlacion de Pearson con Kapak — Agregacion SEMANAL ===')
display(
    summary_weekly_df[['index_column','weeks','kapak_total_sum','index_total_sum','pearson_corr_with_kapak']]
    .round(4)
)

In [ ]:
print('=== Archivos generados ===')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUTPUT_DIR)}')